In [10]:
import pandas as pd
import os
import numpy as np
import spacy
notebook_dir = os.getcwd()
root_dir = os.path.dirname(notebook_dir)
if root_dir not in sys.path:
    sys.path.append(root_dir)


# 2. Import your modular functions from scripts/helpers.py
from scripts.helpers import (
    preprocess_text, 
    analyze_sentiment_batch, 
    extract_top_keywords_per_bank, 
    map_text_to_theme
)


Initializing DistilBERT sentiment pipeline...


Loading weights: 100%|██████████████████████████████████████████████████████████████████████████████████| 104/104 [00:00<00:00, 2516.62it/s]


In [11]:

# Load dynamic absolute paths (built to work both locally and in CI/CD)

INPUT_PATH = os.path.join(BASE_DIR, "../data", "raw", "cleaned_bank_reviews.csv")
OUTPUT_PATH = os.path.join(BASE_DIR, "../data", "raw", "processed_sentiment_reviews.csv")

# Load your cleaned data from Task 1
df = pd.read_csv(INPUT_PATH)
print(f"Loaded {len(df)} reviews successfully.")

Loaded 1827 reviews successfully.


In [12]:
print("Running tokenization, stop-word removal, and lemmatization...")
df['cleaned_text'] = df['review'].apply(preprocess_text)
print("Text preprocessing complete.")
df[['review', 'cleaned_text']].head()

Running tokenization, stop-word removal, and lemmatization...
Text preprocessing complete.


,review,cleaned_text
0,It's not allowing me to transfer money.,allow transfer money
1,IT'S NOT WORK ON HUAWEI DEVICES,work huawei device
2,wow,wow
3,nice app,nice app
4,formative,formative


In [13]:
print("Running DistilBERT deep learning sentiment classification...")
# Extract raw text reviews as a list
raw_reviews = df['review'].tolist()

# Run the batch analyzer defined in your helpers file
raw_results = analyze_sentiment_batch(raw_reviews, batch_size=32)

# Unpack results directly back into your DataFrame
df['sentiment_label'] = [res['label'] for res in raw_results]
df['sentiment_score'] = [res['score'] for res in raw_results]

print("\n--- Sentiment Overview ---")
print(df['sentiment_label'].value_counts())

Running DistilBERT deep learning sentiment classification...

--- Sentiment Overview ---
sentiment_label
NEGATIVE    923
POSITIVE    904
Name: count, dtype: int64


In [14]:
print("Extracting statistically significant keywords using TF-IDF...")

for bank in ['CBE', 'BOA', 'Dashen']:
    print(f"\n--- Top 10 Descriptive Keywords/Phrases for {bank} ---")
    keywords = extract_top_keywords_per_bank(df, 'cleaned_text', bank, top_n=10)
    for word, score in keywords:
        print(f" * {word:<20} (TF-IDF Weight: {score:.4f})")

Extracting statistically significant keywords using TF-IDF...

--- Top 10 Descriptive Keywords/Phrases for CBE ---
 * good                 (TF-IDF Weight: 0.0752)
 * app                  (TF-IDF Weight: 0.0552)
 * nice                 (TF-IDF Weight: 0.0323)
 * work                 (TF-IDF Weight: 0.0243)
 * good app             (TF-IDF Weight: 0.0236)
 * cbe                  (TF-IDF Weight: 0.0181)
 * update               (TF-IDF Weight: 0.0177)
 * thank                (TF-IDF Weight: 0.0169)
 * like                 (TF-IDF Weight: 0.0166)
 * application          (TF-IDF Weight: 0.0164)

--- Top 10 Descriptive Keywords/Phrases for BOA ---
 * app                  (TF-IDF Weight: 0.0597)
 * good                 (TF-IDF Weight: 0.0561)
 * work                 (TF-IDF Weight: 0.0416)
 * bad                  (TF-IDF Weight: 0.0232)
 * bank                 (TF-IDF Weight: 0.0201)
 * good app             (TF-IDF Weight: 0.0197)
 * boa                  (TF-IDF Weight: 0.0168)
 * nice         

In [15]:
# 1. Map reviews into the business themes explained above
df['identified_theme'] = df['review'].apply(map_text_to_theme)

# 2. Reset and build a unique numerical index to fulfill the 'review_id' criteria
df = df.reset_index().rename(columns={'index': 'review_id'})

# 3. Structure the exact column outputs required by Task 2
final_columns = ['review_id', 'review', 'sentiment_label', 'sentiment_score', 'identified_theme']
df_task2_output = df[final_columns].rename(columns={'review': 'review_text'})

# 4. Create output directory if it doesn't exist and save the structured dataset
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
df_task2_output.to_csv(OUTPUT_PATH, index=False)

print(f"Task 2 processing successfully saved to: {OUTPUT_PATH}")
df_task2_output.head()

Task 2 processing successfully saved to: /home/sifen/Desktop/my git/-fintech-review-analytics/notebooks/../data/raw/processed_sentiment_reviews.csv


,review_id,review_text,sentiment_label,sentiment_score,identified_theme
0,0,It's not allowing me to transfer money.,NEGATIVE,0.996865,Transaction Performance
1,1,IT'S NOT WORK ON HUAWEI DEVICES,NEGATIVE,0.999691,General User Feedback
2,2,wow,POSITIVE,0.999592,General User Feedback
3,3,nice app,POSITIVE,0.999806,General User Feedback
4,4,formative,POSITIVE,0.998885,General User Feedback
